In [85]:
import copy
import os
import argparse
from erasure.utils.logger import GLogger
import torch
import numpy as np
import random
from erasure.utils.config.local_ctx import Local
from erasure.utils.config.global_ctx import Global, bcolors 
from erasure.core.factory_base import ConfigurableFactory
from erasure.data.datasets.DatasetManager import DatasetManager

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [86]:
config_file = os.path.join("configs","benchmark", "edge_removal", "Synthetic", "FakeDataset_GAT.jsonc")

In [87]:
global_ctx = Global(config_file)
global_ctx.factory = ConfigurableFactory(global_ctx)

#Create Dataset
data_manager = global_ctx.factory.get_object( Local( global_ctx.config.data ))
global_ctx.dataset = data_manager

@,-823675106 | INFO | 220917 - Creating Global Context for: configs/benchmark/edge_removal/Synthetic/FakeDataset_GAT.jsonc


p��D�,-823675080 | INFO | 220917 - Setting seeds to: 0
!,-823675078 | INFO | 220917 - REMOVAL TYPE SET AS edge
,-823675052 | INFO | 220917 - Caching System: False.
�v��,-823675051 | INFO | 220917 - Created Configurable: erasure.data.preprocessing.preprocess.MakeCentroidFeatures
8�Y|�,-823675051 | INFO | 220917 - Instantiating: torch_geometric.datasets.FakeDataset
{'num_graphs': 1, 'avg_num_nodes': 1000, 'avg_degree': 50, 'num_classes': 2, 'task': 'node'}
�v��,-823675038 | INFO | 220917 - Created Configurable: erasure.data.data_sources.TorchGeometricDataSource.TorchGeometricDataSource
��6��,-823675037 | INFO | 220917 - {'class': 'erasure.data.data_sources.TorchGeometricDataSource.TorchGeometricDataSource', 'parameters': {'datasource': {'class': 'torch_geometric.datasets.FakeDataset', 'parameters': {'num_graphs': 1, 'avg_num_nodes': 1000, 'avg_degree': 50, 'num_classes': 2, 'task': 'node'}}, 'preprocess': [{'class': 'erasure.data.preprocessing.preprocess.MakeCentroidFeatures', 'p

In [88]:
#Create Predictor
current = Local(global_ctx.config.predictor)
current.dataset = data_manager
predictor = global_ctx.factory.get_object(current)
global_ctx.predictor = predictor
global_ctx.logger.info('Global Predictor: ' + str(predictor))

8�Y|�,-823674822 | INFO | 220917 - Instantiating: erasure.model.graphs.GAT.GAT
8�Y|�,-823674819 | INFO | 220917 - Instantiating: torch.optim.Adam
8�Y|�,-823674802 | INFO | 220917 - Instantiating: torch.nn.CrossEntropyLoss
0rO,-823674716 | INFO | 220917 - epoch = 0 ---> loss = 6.8356	 accuracy = 0.0000
0rO,-823674640 | INFO | 220917 - epoch = 1 ---> loss = 4.9261	 accuracy = 0.0000
0rO,-823674560 | INFO | 220917 - epoch = 2 ---> loss = 4.3123	 accuracy = 0.0000
0rO,-823674486 | INFO | 220917 - epoch = 3 ---> loss = 2.6711	 accuracy = 0.0188
0rO,-823674415 | INFO | 220917 - epoch = 4 ---> loss = 2.0486	 accuracy = 0.1962
0rO,-823674334 | INFO | 220917 - epoch = 5 ---> loss = 1.8693	 accuracy = 0.3619
0rO,-823674266 | INFO | 220917 - epoch = 6 ---> loss = 1.4665	 accuracy = 0.4501
0rO,-823674192 | INFO | 220917 - epoch = 7 ---> loss = 1.3083	 accuracy = 0.4712
0rO,-823674121 | INFO | 220917 - epoch = 8 ---> loss = 1.1969	 accuracy = 0.4935
0rO,-823674045 | INFO | 220917 - epoch = 9 ---

In [5]:
data_manager.partitions['all'][0][0]

Data(x=[3327, 3703], edge_index=[2, 9104])

In [6]:
data_manager.partitions['all'][0][1]

tensor([3, 1, 5,  ..., 3, 1, 5])

In [7]:
import torch
print(torch.version.cuda)

12.6


In [8]:
print(predictor.optimizer)

Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    initial_lr: 0.001
    lr: 0.0005000000000000002
    maximize: False
    weight_decay: 0
)


In [9]:
import torch
print(torch.__version__)
print(torch.version.cuda)


2.7.0+cu126
12.6


In [10]:
#Create unlearners 
unlearners = []
unlearners_cfg = global_ctx.config.unlearners
for un in unlearners_cfg:
    current = Local(un)
    current.dataset = data_manager
    current.predictor = copy.deepcopy(predictor)
    unlearners.append( global_ctx.factory.get_object(current) )

data manager <erasure.data.datasets.DatasetManager.DatasetManager object at 0x7fc0120c5820>
�|�/�,1677990031 | INFO | 1226080 - Created Configurable: erasure.unlearners.GoldModel.GoldModelGraph
�{��,1677990222 | INFO | 1226080 - Instantiating: torch.optim.Adam
�|�/�,1677990226 | INFO | 1226080 - Created Configurable: erasure.unlearners.graph_unlearners.UNSIR.UNSIR
�{��,1677990288 | INFO | 1226080 - Instantiating: torch.optim.Adam
�|�/�,1677990289 | INFO | 1226080 - Created Configurable: erasure.unlearners.graph_unlearners.SelectiveSynapticDampening.SelectiveSynapticDampening
�|�/�,1677990332 | INFO | 1226080 - Created Configurable: erasure.unlearners.composite.Identity


/NFSHOME/adangelo/LinkInference/erasure/unlearners/graph_unlearners/GraphUnlearner.py:24: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.labels = torch.tensor(self.labels)


In [11]:
data_manager.partitions['all'][0][0]

Data(x=[3327, 3703], edge_index=[2, 9104])

In [12]:
data_manager.partitions['all'][0][0].edge_index

tensor([[   0,    1,    1,  ..., 3324, 3325, 3326],
        [ 628,  158,  486,  ..., 2820, 1643,   33]])

In [13]:
data_manager.partitions['all'].revise_graph_edges([(0,1),(0,2)])[0][0]

Data(x=[3327, 3703], edge_index=[2, 0])

In [14]:
print(len( data_manager.partitions['forget']) )

1820


In [15]:
#Evaluator
current = Local(global_ctx.config.evaluator)
current.unlearners = unlearners
evaluator = global_ctx.factory.get_object(current)

# Evaluations
for unlearner in unlearners:
    global_ctx.logger.info(f'''{bcolors.OKGREEN}####\t\t Evaluating Unlearner {unlearner.__class__.__name__} \t\t####{bcolors.ENDC}''')
    evaluator.evaluate(unlearner,predictor)

�|�/�,1677990975 | INFO | 1226080 - Created Configurable: erasure.evaluations.running.RunTime
�|�/�,1677991069 | INFO | 1226080 - Created Configurable: erasure.evaluations.measures.SaveValues
�|�/�,1677991070 | INFO | 1226080 - Created Configurable: erasure.evaluations.manager.Evaluator
pJ�e�,1677991070 | INFO | 1226080 - ####		 Evaluating Unlearner GoldModelGraph 		####
�}y/�,1677991106 | INFO | 1226080 - Unlearning copyed predictor: <erasure.model.TorchGraphModel.TorchGraphModel object at 0x7fc1c41c0250>
�{��,1677991107 | INFO | 1226080 - Instantiating: erasure.model.graphs.GCN.GCN
�{��,1677991116 | INFO | 1226080 - Instantiating: torch.optim.Adam
�{��,1677991116 | INFO | 1226080 - Instantiating: torch.nn.CrossEntropyLoss
��t,1677991173 | INFO | 1226080 - epoch = 0 ---> loss = 1.7887	 accuracy = 0.1871
��t,1677991225 | INFO | 1226080 - epoch = 1 ---> loss = 1.7329	 accuracy = 0.3812
��t,1677991277 | INFO | 1226080 - epoch = 2 ---> loss = 1.6814	 accuracy = 0.5027
��t,16779